# InterPro Scan API

In [ ]:
mkdir my_interproscan
cd my_interproscan
wget https://ftp.ebi.ac.uk/pub/software/unix/iprscan/5/5.76-107.0/interproscan-5.76-107.0-64-bit.tar.gz
wget https://ftp.ebi.ac.uk/pub/software/unix/iprscan/5/5.76-107.0/interproscan-5.76-107.0-64-bit.tar.gz.md5

# Recommended checksum to confirm the download was successful:
md5sum -c interproscan-5.76-107.0-64-bit.tar.gz.md5
# Must return *interproscan-5.76-107.0-64-bit.tar.gz: OK*
# If not - try downloading the file again as it may be a corrupted copy.

tar -pxvzf interproscan-5.76-107.0-*-bit.tar.gz

python3 setup.py -f interproscan.properties

In [2]:
cd my_interproscan

data/my_interproscan


IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [8]:
cd interproscan-5.76-107.0/

data/my_interproscan/interproscan-5.76-107.0


In [ ]:
python3 setup.py -f interproscan.properties

In [ ]:
interproscan.sh -i test_all_appl.fasta -f tsv

## Script to setup Interpro

In [ ]:
mkdir my_interproscan

cd my_interproscan

wget https://ftp.ebi.ac.uk/pub/software/unix/iprscan/5/5.76-107.0/interproscan-5.76-107.0-64-bit.tar.gz
wget https://ftp.ebi.ac.uk/pub/software/unix/iprscan/5/5.76-107.0/interproscan-5.76-107.0-64-bit.tar.gz.md5

# Recommended checksum to confirm the download was successful:
md5sum -c interproscan-5.76-107.0-64-bit.tar.gz.md5
# Must return *interproscan-5.76-107.0-64-bit.tar.gz: OK*
# If not - try downloading the file again as it may be a corrupted copy.

tar -pxvzf interproscan-5.76-107.0-*-bit.tar.gz

cd interproscan-5.76-107.0

python3 setup.py -f interproscan.properties

## Run Interproscan on a protein sequence

In [ ]:
#!/usr/bin/env python3

import argparse
import pandas as pd
import subprocess
import os
import sys


TSV_COLS = [
    "protein", "md5", "length", "analysis", "signature_acc", "signature_desc",
    "start", "end", "score", "status", "date",
    "interpro_id", "interpro_desc", "go_terms", "pathway"
]


def run_interproscan(interproscan_path, fasta_path, cpu_num, prefix):
    """Run InterProScan locally and return path to TSV output."""
    tsv_output = f"{prefix}.tsv"

    cmd = [
        interproscan_path,
        "-i", fasta_path,
        "-f", "tsv",
        "-o", tsv_output,
        '-cpu', str(cpu_num)
    ]

    print(f"\n➡️ Running InterProScan:\n   {' '.join(cmd)}\n", file=sys.stderr)
    subprocess.run(cmd, check=True)

    if not os.path.exists(tsv_output):
        raise FileNotFoundError(f"❌ Could not find InterProScan output: {tsv_output}")

    return tsv_output


def parse_and_collapse(tsv_path, out_domains):
    """Parse InterProScan TSV + collapse duplicate InterPro IDs."""

    # Load
    df = pd.read_csv(tsv_path, sep="\t", header=None, comment="#")

    # Fix columns depending on length
    ncols = df.shape[1]
    cols = TSV_COLS[:ncols] + [f"extra_{i}" for i in range(ncols - len(TSV_COLS))]
    df.columns = cols

    # Keep rows with actual InterPro IDs
    df = df[df["interpro_id"].notna() & (df["interpro_id"] != "-")]

    # Convert positions
    df["start"] = pd.to_numeric(df["start"], errors="coerce")
    df["end"] = pd.to_numeric(df["end"], errors="coerce")

    # =============== COLLAPSE LOGIC ===============
    # Group by protein + InterPro ID and collapse fragments
    grouped = df.groupby(
        ["protein", "interpro_id", "interpro_desc"],
        dropna=False
    ).agg(
        start=("start", "min"),
        end=("end", "max"),
        n_fragments=("start", "count")
    ).reset_index()

    # Final schema
    grouped.rename(columns={
        "protein": "accession",
        "interpro_desc": "entry_name"
    }, inplace=True)

    # Drop analysis/type column as requested
    # (It was only used to distinguish sources; now unnecessary)
    # Nothing extra to drop since we already excluded it.

    # Write output
    grouped.to_csv(out_domains, sep="\t", index=False)
    print(f"✅ Saved domains: {out_domains}")


def main():
    parser = argparse.ArgumentParser(description="Run local InterProScan and collapse domain fragments.")
    parser.add_argument("fasta", help="Path to FASTA file")
    parser.add_argument("interproscan", help="Path to interproscan.sh")
    parser.add_argument("--prefix", default="interpro_local", help="Output prefix")
    parser.add_argument("--cpu", type=int, default=1, help="Number of CPUs to use")
    args = parser.parse_args()

    # Run interproscan
    tsv_path = run_interproscan(args.interproscan, args.fasta, args.cpu, args.prefix)

    # Parse + collapse
    parse_and_collapse(
        tsv_path,
        out_domains=f"{args.prefix}_domains.tsv"
    )


if __name__ == "__main__":
    main()

In [ ]:
python3 interpro_run.py \
    my_interproscan/interproscan-5.76-107.0/test_all_appl.fasta \
    my_interproscan/interproscan-5.76-107.0/interproscan.sh \
    --prefix test_run --cpu 8

# Getting InterPros for Proteins in the Test set that are missing Interpro IDs

In [1]:
import pandas as pd
from datasets import load_dataset

# Load CAFA5 dataset
dataset = load_dataset("wanglab/cafa5", "temporal_holdout_ext_2022_2025_reasoning")

test_df_proteins = dataset['test'].to_pandas()

In [2]:
test_df_proteins.columns

Index(['protein_id', 'protein_names', 'protein_function', 'organism',
       'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc',
       'go_bp_created', 'go_mf_created', 'go_cc_created', 'date_added',
       'last_modified', 'length', 'structure_path', 'interpro_ids',
       'interpro_location', 'string_id', 'interaction_partners',
       'ppi_formatted'],
      dtype='object')

In [3]:
test_df_proteins

,protein_id,protein_names,protein_function,organism,subcellular_location,sequence,go_ids,go_bp,go_mf,go_cc,...,go_cc_created,date_added,last_modified,length,structure_path,interpro_ids,interpro_location,string_id,interaction_partners,ppi_formatted
0,P0DO85,Strychnine-10-hydroxylase,Monooxygenase involved in the biosynthesis of ...,Strychnos nux-vomica (Poison nut) (Strychnine ...,Membrane; Single-pass membrane protein,MEFSLLYIHTAILGLISLFLILHFVFWRLKSAKGGSAKNSLPPEAG...,['GO:0008150' 'GO:0008152' 'GO:0009058' 'GO:00...,['GO:0008150' 'GO:0008152' 'GO:0009058' 'GO:00...,['GO:0003674' 'GO:0003824' 'GO:0004497' 'GO:00...,None,...,NaN,2024-11-27,2025-06-18,524,AF-P0DO85-F1-model_v6.pdb,"[IPR050651, IPR017972, IPR001128, IPR002401, I...","{""IPR036396"": [33, 518], ""IPR050651"": [19, 524...",None,None,None
1,P0DO87,Strychnine-11-hydroxylase,Monooxygenase involved in the biosynthesis of ...,Strychnos nux-vomica (Poison nut) (Strychnine ...,Membrane; Single-pass membrane protein,MHSAMSFLLLFSLCFLIHCFVFLLIKKKKAKTMDAKTVPPGPKKLP...,['GO:0008150' 'GO:0008152' 'GO:0009058' 'GO:00...,['GO:0008150' 'GO:0008152' 'GO:0009058' 'GO:00...,['GO:0003674' 'GO:0003824' 'GO:0004497' 'GO:00...,None,...,NaN,2024-11-27,2025-06-18,508,AF-P0DO87-F1-model_v6.pdb,"[IPR017972, IPR036396, IPR001128, IPR002401]","{""IPR036396"": [23, 500], ""IPR001128"": [39, 489...",None,None,None
2,P46077,14-3-3-like protein GF14 phi,Is associated with a DNA binding complex that ...,Arabidopsis thaliana (Mouse-ear cress),Nucleus. Cytoplasm. Note=Translocates from the...,MAAPPASSSAREEFVYLAKLAEQAERYEEMVEFMEKVAEAVDKDEL...,['GO:0007154' 'GO:0007165' 'GO:0007166' 'GO:00...,['GO:0007154' 'GO:0007165' 'GO:0007166' 'GO:00...,None,None,...,NaN,1995-11-01,2025-06-18,267,AF-P46077-F1-model_v6.pdb,"[IPR023410, IPR036815, IPR023409, IPR000308]","{""IPR036815"": [2, 254], ""IPR000308"": [8, 251],...",3702.P46077,"[Q9LK96, Q9LYJ6, O49516, Q9LZE1, Q9ZRZ8, P1945...","[AT3g15090/K15M2_24, Shaggy-related protein ki..."
3,P48348,14-3-3-like protein G-BOX factor 14 kappa,Is associated with a DNA binding complex that ...,Arabidopsis thaliana (Mouse-ear cress),Nucleus. Cytoplasm. Note=Translocates from the...,MATTLSRDQYVYMAKLAEQAERYEEMVQFMEQLVSGATPAGELTVE...,['GO:0005575' 'GO:0005622' 'GO:0005634' 'GO:00...,None,None,['GO:0005575' 'GO:0005622' 'GO:0005634' 'GO:00...,...,20221025.0,1996-02-01,2025-06-18,248,AF-P48348-F1-model_v6.pdb,"[IPR023410, IPR036815, IPR023409, IPR000308]","{""IPR036815"": [1, 248], ""IPR000308"": [4, 246],...",3702.P48348,"[Q9C9Q2, Q9FZ61, Q9LZE1, Q9ZWB6, Q9ZTA4, Q9628...","[Protein BRASSINAZOLE-RESISTANT 1, Serine/thre..."
4,Q96299,14-3-3-like protein GF14 mu,Is associated with a DNA binding complex that ...,Arabidopsis thaliana (Mouse-ear cress),Nucleus. Cytoplasm. Note=Translocates from the...,MGSGKERDTFVYLAKLSEQAERYEEMVESMKSVAKLNVDLTVEERN...,['GO:0007275' 'GO:0008150' 'GO:0009791' 'GO:00...,['GO:0007275' 'GO:0008150' 'GO:0009791' 'GO:00...,None,None,...,NaN,1998-07-15,2025-06-18,263,AF-Q96299-F1-model_v6.pdb,"[IPR023410, IPR036815, IPR023409, IPR000308]","{""IPR036815"": [1, 247], ""IPR000308"": [4, 244],...",3702.Q96299,"[Q4V397, Q9SRI1, Q93YR3, F4K1T0, Q9ZS06, Q9M2P...","[Ankyrin repeat family protein, Protein transp..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17810,A0A6I8RXM5,Mothers against decapentaplegic homolog,None,Xenopus tropicalis (Western clawed frog) (Silu...,Cytoplasm. Nucleus,MFRTKRSVLVRRLWRSRAPGESGEGEGGERVHGAGGCCLGKSANKV...,['GO:0001702' 'GO:0003401' 'GO:0007275' 'GO:00...,['GO:0001702' 'GO:0003401' 'GO:0007275' 'GO:00...,None,None,...,NaN,2020-12-02,2025-10-08,357,AF-A0A6I8RXM5-F1-model_v6.pdb,"[IPR036578, IPR008984, IPR013790, IPR001132, I...","{""IPR036578"": [9, 162], ""IPR008984"": [161, 356...",None,None,None
17811,F7EN59,Secreted frizzled-related protein 1,None,Xenopus tropicalis (Western clawed frog) (Silu...,Cell membrane; Multi-pass membrane protein. Se...,MNSEGGIWPLLLFWVTPGILSQAPQASEYDYVSFQPDLGRQYQSGR

In [4]:
print(
    f"""
Missing values:
---------------
protein_id:           {test_df_proteins['protein_id'].isna().sum()}
protein_names:        {test_df_proteins['protein_names'].isna().sum()}
protein_function:     {test_df_proteins['protein_function'].isna().sum()}
organism:             {test_df_proteins['organism'].isna().sum()}
subcellular_location: {test_df_proteins['subcellular_location'].isna().sum()}
go_ids:               {test_df_proteins['go_ids'].isna().sum()}
go_bp:                {test_df_proteins['go_bp'].isna().sum()}
go_mf:                {test_df_proteins['go_mf'].isna().sum()}
go_cc:                {test_df_proteins['go_cc'].isna().sum()}
sequence:             {test_df_proteins['sequence'].isna().sum()}
go_bp_created:        {test_df_proteins['go_bp_created'].isna().sum()}
go_mf_created:        {test_df_proteins['go_mf_created'].isna().sum()}
go_cc_created:        {test_df_proteins['go_cc_created'].isna().sum()}
date_added:           {test_df_proteins['date_added'].isna().sum()}
last_modified:        {test_df_proteins['last_modified'].isna().sum()}
"""
)


Missing values:
---------------
protein_id:           0
protein_names:        0
protein_function:     9381
organism:             0
subcellular_location: 5344
go_ids:               0
go_bp:                7207
go_mf:                11709
go_cc:                10661
sequence:             0
go_bp_created:        7207
go_mf_created:        11709
go_cc_created:        10661
date_added:           45
last_modified:        0



In [5]:
test_df_proteins['interpro_ids']

0        [IPR050651, IPR017972, IPR001128, IPR002401, I...
1             [IPR017972, IPR036396, IPR001128, IPR002401]
2             [IPR023410, IPR036815, IPR023409, IPR000308]
3             [IPR023410, IPR036815, IPR023409, IPR000308]
4             [IPR023410, IPR036815, IPR023409, IPR000308]
                               ...                        
17810    [IPR036578, IPR008984, IPR013790, IPR001132, I...
17811    [IPR008993, IPR036790, IPR001134, IPR015526, I...
17812                                          [IPR036514]
17813                               [IPR004023, IPR036605]
17814    [IPR001995, IPR000477, IPR043502, IPR034170, I...
Name: interpro_ids, Length: 17815, dtype: object

In [6]:
import numpy as np 

empty_or_na = test_df_proteins[
    test_df_proteins['interpro_ids'].isna() | 
    test_df_proteins['interpro_ids'].apply(lambda x: isinstance(x, np.ndarray) and len(x) == 0)
]
empty_or_na

,protein_id,protein_names,protein_function,organism,subcellular_location,sequence,go_ids,go_bp,go_mf,go_cc,...,go_cc_created,date_added,last_modified,length,structure_path,interpro_ids,interpro_location,string_id,interaction_partners,ppi_formatted
156,P46007,AAF/I fimbrial subunit,None,Escherichia coli,Fimbrium,MKTLKNMRRKNLCITLGLVSLLSRGANAALERPPIKATETIRLTVT...,['GO:0008150' 'GO:0044403' 'GO:0044406' 'GO:00...,['GO:0008150' 'GO:0044403' 'GO:0044406' 'GO:00...,None,None,...,NaN,1995-11-01,2025-06-18,171,AF-P46007-F1-model_v6.pdb,None,NaN,None,None,None
176,Q9LYF6,Arabinogalactan protein 15,Proteoglycan that seems to be implicated in di...,Arabidopsis thaliana (Mouse-ear cress),"Cell membrane; Lipid-anchor, GPI- anchor",MAISKASIVVLMMVIISVVASAQSEAPAPSPTSGSSAISASFVSAG...,['GO:0005575' 'GO:0016020' 'GO:0110165'],None,None,['GO:0005575' 'GO:0016020' 'GO:0110165'],...,20220825.0,2006-12-12,2025-06-18,61,AF-Q9LYF6-F1-model_v6.pdb,None,NaN,3702.Q9LYF6,"[Q9ZT18, Q94A05, W8QP19, Q9ZT16, Q9S7N4, Q9LVC...","[Classical arabinogalactan protein 2, Hydroxyp..."
184,Q5PP12,Arabinogalactan protein 24,Proteoglycan that seems to be implicated in di...,Arabidopsis thaliana (Mouse-ear cress),"Cell membrane; Lipid-anchor, GPI- anchor",MMMMTKMFVQIAVVCLLATMAVVSAHEGHHHHAPAPAPGPASSSTV...,['GO:0005575' 'GO:0016020' 'GO:0110165'],None,None,['GO:0005575' 'GO:0016020' 'GO:0110165'],...,20220825.0,2006-12-12,2025-06-18,69,AF-Q5PP12-F1-model_v6.pdb,None,NaN,3702.Q5PP12,"[Q9FK16, Q8L9P8, Q9LYF6, Q9SVT5, Q941E9, Q8LCR...","[Arabinogalactan protein 22, Protein RALF-like..."
187,Q9SQT9,Classical arabinogalactan protein 27,Proteoglycan that seems to be implicated in di...,Arabidopsis thaliana (Mouse-ear cress),"Cell membrane; Lipid-anchor, GPI- anchor",MASSILLTLITFIFLSSLSLSSPTTNTIPSSQTISPSEEKISPEIA...,['GO:0005575' 'GO:0016020' 'GO:0110165'],None,None,['GO:0005575' 'GO:0016020' 'GO:0110165'],...,20220825.0,2006-12-12,2025-04-09,125,AF-Q9SQT9-F1-model_v6.pdb,None,NaN,3702.Q9SQT9,"[Q9SX61, O81069, Q9XH62, Q9M0S4, Q94F57, Q9ZT1...",[CBL-interacting serine/threonine-protein kina...
261,Q9UTK5,Nucleoporin alm1,Maintains the proteasome and its anchor cut8 a...,Schizosaccharomyces pombe (strain 972 / ATCC 2...,"Nucleus, nuclear pore complex. Nucleus envelope",MSSGGLEDDIQLVHEFLDVSFEDIKPLVSVNGFAVFISAIKTKVKD...,['GO:0008150' 'GO:0009893' 'GO:0010564' 'GO:00...,['GO:0008150' 'GO:0009893' 'GO:0010564' 'GO:00...,None,None,...,NaN,2001-08-14,2025-06-18,1727,AF-Q9UTK5-F1-model_v6.pdb,None,NaN,284812.Q9UTK5,"[O74424, Q10331, O13838, Q10099, O74315, Q9UTH...","[Nucleoporin nup211, Nucleoporin nup107, Nucle..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17055,A0A8M9P6P2,PHD finger protein 21A isoform X5,None,Danio rerio (Zebrafish) (Brachydanio rerio),None,MNVCNTASVSPTVTWRGCSSESSDGRANDPAHLLHCHKFLSSGPLL...,['GO:0008150' 'GO:0032502' 'GO:0048856' 'GO:00...,['GO:0008150' 'GO:0032502' 'GO:0048856' 'GO:00...,None,None,...,NaN,2022-08-03,2025-10-08,840,AF-A0A8M9P6P2-F1-model_v6.pdb,None,NaN,None,None,None
17384,Q7SXY5,Pum1 protein,None,Danio rerio (Zebrafish) (Brachydanio rerio),None,MSVACVLKRKAVLWQDSFSPHLRLPPPDRPLPSMPVVLSAAGHTPQ...,['GO:0003674' 'GO:0003676' 'GO:0003723' 'GO:00...,None,['GO:0003674' 'GO:0003676' 'GO:0003723' 'GO:00...,None,...,NaN,2003-10-01,2025-10-08,540,AF-Q7SXY5-F1-model_v6.pdb,None,NaN,None,None,None
17481,A0A8J0TMV4,Uncharacterized protein tmem79.L isoform X2,None,Xenopus laevis (African clawed frog),None,MVSPEAPEKQRDVDSGVVSDSGDLPIVVAQPNMNVDKARPIKSGGS...,['GO:0007154' 'GO:0007165' 'GO:0007166' 'GO:00...,['GO:0007154' 'GO:0007165' 'GO:0007166' 'GO:00...,None,None,...,NaN,2022-05-25,2025-10-08,476,AF-A0A8J0TMV4-F1-model_v6.pdb,None,NaN,None,None,None
17513,A0A974CA66,Transmembrane protein 79,None,Xenopus laevis (African clawed frog),None,MVSPEAPEKQRDVDSGVVSDSGDLPIVVAQPNMNVDKARPIKSGGS...,['GO:0007154' 'GO:0007165' 'GO:0007166' 'GO:00...,['GO:0007154' 'GO:0007165' 'GO:0007166' 'GO:00...,None,None,...,NaN,2023-02-22,2025-04-02

In [20]:
with open('temporal-holdout-ext-proteins-wo-interpro.fasta', 'w') as f:
    for idx, row in empty_or_na.iterrows():
        f.write('>' + row['protein_id'] + '\n')
        f.write(row['sequence'] + '\n')

In [ ]:
python3 interpro_run.py \
    temporal-holdout-ext-proteins-wo-interpro.fasta \
    my_interproscan/interproscan-5.76-107.0/interproscan.sh \
    --prefix proteins_wo_interpro --cpu 8

In [7]:
protein_wo_interpro = pd.read_csv('temporal-holdout-ext-proteins-wo-interpro_domains.tsv', sep='\t')
protein_wo_interpro

,accession,interpro_id,entry_name,start,end,n_fragments
0,A0A096MJE9,IPR011990,Tetratricopeptide-like helical domain superfamily,1,147,2
1,A0A096MJE9,IPR019734,Tetratricopeptide repeat,1,67,4
2,A0A096MK21,IPR042887,Vesicle-associated membrane protein 4,1,40,1
3,A0A0A0MRK1,IPR000467,G-patch domain,9,57,3
4,A0A0A0MRK1,IPR050656,PIN2/TERF1-interacting telomerase inhibitor,1,348,1
...,...,...,...,...,...,...
238,Q4KLH7,IPR006910,"Rad21/Rec8-like protein, N-terminal",1,103,1
239,Q4KLH7,IPR023093,"ScpA-like, C-terminal",561,629,1
240,Q4KLH7,IPR036390,Winged helix DNA-binding domain superfamily,562,632,1
241,Q4KLH7,IPR039781,Rad21/Rec8-like protein,1,632,1


In [8]:
len(protein_wo_interpro['accession'].drop_duplicates())

45

In [9]:
from collections import defaultdict

# Intermediate mapping
accession_to_entries = defaultdict(list)

# Store everything per accession
# Now only 4 fields: acc, ipr, start, end
for acc, ipr, start, end in zip(
    protein_wo_interpro["accession"],
    protein_wo_interpro["interpro_id"],
    protein_wo_interpro["start"],
    protein_wo_interpro["end"]
):
    accession_to_entries[acc].append((ipr, start, end))

# Final output structure
collapsed_data = []

for acc, entries in accession_to_entries.items():
    interpro_ids = set()
    interpro_location = {}

    # Sort ONLY by start coordinate
    entries_sorted = sorted(entries, key=lambda x: x[1])  # x[1] = start

    for ipr, start, end in entries_sorted:
        # If same IPR appears multiple times → collapse using min/max
        if ipr in interpro_location:
            prev_start, prev_end = interpro_location[ipr]
            interpro_location[ipr] = (
                min(prev_start, start),
                max(prev_end, end)
            )
        else:
            interpro_location[ipr] = (start, end)

        interpro_ids.add(ipr)

    collapsed_data.append({
        "accession": acc,
        "interpro_ids": list(interpro_ids),
        "interpro_location": interpro_location
    })

# Convert to DataFrame
collapsed_df = pd.DataFrame(collapsed_data)

In [10]:
collapsed_df

,accession,interpro_ids,interpro_location
0,A0A096MJE9,"[IPR019734, IPR011990]","{'IPR011990': (1, 147), 'IPR019734': (1, 67)}"
1,A0A096MK21,[IPR042887],"{'IPR042887': (1, 40)}"
2,A0A0A0MRK1,"[IPR000467, IPR050656]","{'IPR050656': (1, 348), 'IPR000467': (9, 57)}"
3,A0A0G2JUI0,"[IPR015224, IPR035964, IPR035963, IPR002404, I...","{'IPR032425': (6, 87), 'IPR029071': (83, 139),..."
4,A0A0G2JWY7,"[IPR001356, IPR046333, IPR017970, IPR009057, I...","{'IPR046333': (3, 94), 'IPR009057': (8, 80), '..."
5,A0A0G2K2R3,"[IPR009057, IPR001356, IPR046333, IPR017970]","{'IPR001356': (1, 40), 'IPR046333': (1, 52), '..."
6,A0A0G2K363,"[IPR001356, IPR013087, IPR009057, IPR008598, I...","{'IPR051574': (137, 1064), 'IPR008598': (187, ..."
7,A0A0G2K7M0,"[IPR053896, IPR007110, IPR003599, IPR013783, I...","{'IPR007110': (7, 237), 'IPR050504': (18, 282)..."
8,A0A140TAA1,"[IPR051102, IPR007110, IPR003599, IPR013783, I...","{'IPR051102': (21, 612), 'IPR036179': (29, 554..."
9,A0A8I5Y7L2,"[IPR027417, IPR014001, IPR000629, IPR014014, I...","{'IPR027417': (35, 436), 'IPR014014': (69, 97)..."


# Update Train and Test with Interpro Data

In [11]:
interpro_collapsed = collapsed_df.rename(columns={"accession": "protein_id"})

# Ensure both DFs use protein_id as index
df_main = test_df_proteins.set_index("protein_id")
df_new  = interpro_collapsed.set_index("protein_id")

# Update existing cells with new values (does NOT drop anything else!)
df_main.update(df_new)

# Restore index as column if needed
test_df_proteins_updated = df_main.reset_index()

In [12]:
print("Rows where test interpro is None:", test_df_proteins_updated["interpro_ids"].isnull().sum())

Rows where test interpro is None: 755


In [13]:
test_df_proteins_updated

,protein_id,protein_names,protein_function,organism,subcellular_location,sequence,go_ids,go_bp,go_mf,go_cc,...,go_cc_created,date_added,last_modified,length,structure_path,interpro_ids,interpro_location,string_id,interaction_partners,ppi_formatted
0,P0DO85,Strychnine-10-hydroxylase,Monooxygenase involved in the biosynthesis of ...,Strychnos nux-vomica (Poison nut) (Strychnine ...,Membrane; Single-pass membrane protein,MEFSLLYIHTAILGLISLFLILHFVFWRLKSAKGGSAKNSLPPEAG...,['GO:0008150' 'GO:0008152' 'GO:0009058' 'GO:00...,['GO:0008150' 'GO:0008152' 'GO:0009058' 'GO:00...,['GO:0003674' 'GO:0003824' 'GO:0004497' 'GO:00...,None,...,NaN,2024-11-27,2025-06-18,524,AF-P0DO85-F1-model_v6.pdb,"[IPR050651, IPR017972, IPR001128, IPR002401, I...","{""IPR036396"": [33, 518], ""IPR050651"": [19, 524...",None,None,None
1,P0DO87,Strychnine-11-hydroxylase,Monooxygenase involved in the biosynthesis of ...,Strychnos nux-vomica (Poison nut) (Strychnine ...,Membrane; Single-pass membrane protein,MHSAMSFLLLFSLCFLIHCFVFLLIKKKKAKTMDAKTVPPGPKKLP...,['GO:0008150' 'GO:0008152' 'GO:0009058' 'GO:00...,['GO:0008150' 'GO:0008152' 'GO:0009058' 'GO:00...,['GO:0003674' 'GO:0003824' 'GO:0004497' 'GO:00...,None,...,NaN,2024-11-27,2025-06-18,508,AF-P0DO87-F1-model_v6.pdb,"[IPR017972, IPR036396, IPR001128, IPR002401]","{""IPR036396"": [23, 500], ""IPR001128"": [39, 489...",None,None,None
2,P46077,14-3-3-like protein GF14 phi,Is associated with a DNA binding complex that ...,Arabidopsis thaliana (Mouse-ear cress),Nucleus. Cytoplasm. Note=Translocates from the...,MAAPPASSSAREEFVYLAKLAEQAERYEEMVEFMEKVAEAVDKDEL...,['GO:0007154' 'GO:0007165' 'GO:0007166' 'GO:00...,['GO:0007154' 'GO:0007165' 'GO:0007166' 'GO:00...,None,None,...,NaN,1995-11-01,2025-06-18,267,AF-P46077-F1-model_v6.pdb,"[IPR023410, IPR036815, IPR023409, IPR000308]","{""IPR036815"": [2, 254], ""IPR000308"": [8, 251],...",3702.P46077,"[Q9LK96, Q9LYJ6, O49516, Q9LZE1, Q9ZRZ8, P1945...","[AT3g15090/K15M2_24, Shaggy-related protein ki..."
3,P48348,14-3-3-like protein G-BOX factor 14 kappa,Is associated with a DNA binding complex that ...,Arabidopsis thaliana (Mouse-ear cress),Nucleus. Cytoplasm. Note=Translocates from the...,MATTLSRDQYVYMAKLAEQAERYEEMVQFMEQLVSGATPAGELTVE...,['GO:0005575' 'GO:0005622' 'GO:0005634' 'GO:00...,None,None,['GO:0005575' 'GO:0005622' 'GO:0005634' 'GO:00...,...,20221025.0,1996-02-01,2025-06-18,248,AF-P48348-F1-model_v6.pdb,"[IPR023410, IPR036815, IPR023409, IPR000308]","{""IPR036815"": [1, 248], ""IPR000308"": [4, 246],...",3702.P48348,"[Q9C9Q2, Q9FZ61, Q9LZE1, Q9ZWB6, Q9ZTA4, Q9628...","[Protein BRASSINAZOLE-RESISTANT 1, Serine/thre..."
4,Q96299,14-3-3-like protein GF14 mu,Is associated with a DNA binding complex that ...,Arabidopsis thaliana (Mouse-ear cress),Nucleus. Cytoplasm. Note=Translocates from the...,MGSGKERDTFVYLAKLSEQAERYEEMVESMKSVAKLNVDLTVEERN...,['GO:0007275' 'GO:0008150' 'GO:0009791' 'GO:00...,['GO:0007275' 'GO:0008150' 'GO:0009791' 'GO:00...,None,None,...,NaN,1998-07-15,2025-06-18,263,AF-Q96299-F1-model_v6.pdb,"[IPR023410, IPR036815, IPR023409, IPR000308]","{""IPR036815"": [1, 247], ""IPR000308"": [4, 244],...",3702.Q96299,"[Q4V397, Q9SRI1, Q93YR3, F4K1T0, Q9ZS06, Q9M2P...","[Ankyrin repeat family protein, Protein transp..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17810,A0A6I8RXM5,Mothers against decapentaplegic homolog,None,Xenopus tropicalis (Western clawed frog) (Silu...,Cytoplasm. Nucleus,MFRTKRSVLVRRLWRSRAPGESGEGEGGERVHGAGGCCLGKSANKV...,['GO:0001702' 'GO:0003401' 'GO:0007275' 'GO:00...,['GO:0001702' 'GO:0003401' 'GO:0007275' 'GO:00...,None,None,...,NaN,2020-12-02,2025-10-08,357,AF-A0A6I8RXM5-F1-model_v6.pdb,"[IPR036578, IPR008984, IPR013790, IPR001132, I...","{""IPR036578"": [9, 162], ""IPR008984"": [161, 356...",None,None,None
17811,F7EN59,Secreted frizzled-related protein 1,None,Xenopus tropicalis (Western clawed frog) (Silu...,Cell membrane; Multi-pass membrane protein. Se...,MNSEGGIWPLLLFWVTPGILSQAPQASEYDYVSFQPDLGRQYQSGR

## Update the interpro metadata

In [17]:
import pandas as pd
from datasets import load_dataset

# Load CAFA5 dataset
interpro_dataset = load_dataset("wanglab/cafa5", "interpro_temp_holdout_ext_22_25_metadata")

interpro_metadata = interpro_dataset['metadata'].to_pandas()

## Interpro Metadata

In [ ]:
interpro_metadata

,interpro_id,entry_name,type
0,IPR001523,Paired domain,domain
1,IPR009057,Homedomain-like superfamily,homologous_superfamily
2,IPR022130,Paired-box protein 2 C-terminal,domain
3,IPR036388,Winged helix-like DNA-binding domain superfamily,homologous_superfamily
4,IPR043182,Paired DNA-binding domain,domain
...,...,...,...
14181,IPR000332,Beta 2 adrenoceptor,family
14182,IPR002233,Adrenoceptor family,family
14183,IPR009976,Exocyst complex component Sec10-like,family
14184,IPR048625,"Exocyst complex component Sec10, N-terminal",domain


In [19]:
protein_wo_interpro[['interpro_id', 'entry_name']].drop_duplicates()

,interpro_id,entry_name
0,IPR011990,Tetratricopeptide-like helical domain superfamily
1,IPR019734,Tetratricopeptide repeat
2,IPR042887,Vesicle-associated membrane protein 4
3,IPR000467,G-patch domain
4,IPR050656,PIN2/TERF1-interacting telomerase inhibitor
...,...,...
237,IPR006909,"Rad21/Rec8-like protein, C-terminal, eukaryotic"
238,IPR006910,"Rad21/Rec8-like protein, N-terminal"
239,IPR023093,"ScpA-like, C-terminal"
241,IPR039781,Rad21/Rec8-like protein


In [22]:
interpro_metadata_updated = pd.concat([interpro_metadata, protein_wo_interpro[['interpro_id', 'entry_name']].drop_duplicates()])

**QC to check if every interpro ID in test reasoning is in this metadata df**

In [23]:
# Step 1: create a set of valid InterPro IDs
valid_ids = set(interpro_metadata_updated["interpro_id"].dropna())

# Step 2: explode interpro_ids into separate rows
exploded = test_df_proteins_updated[["interpro_ids"]].explode("interpro_ids")

# Step 3: filter out null values first
exploded = exploded[exploded["interpro_ids"].notna()]

# Step 4: mark invalid IDs
exploded["is_invalid"] = ~exploded["interpro_ids"].isin(valid_ids)

# Step 5: count invalids
invalid_count = exploded["is_invalid"].sum()

print(invalid_count if invalid_count > 0 else 0)

0


**All interpro ids in test dataframes are in this metadata df**

# Upload Data

In [24]:
import json

In [25]:
#Making it into a json object so that huggingface can handle it
test_df_proteins_updated["interpro_location"] = test_df_proteins_updated["interpro_location"].apply(json.dumps)

In [26]:
from datasets import Dataset

test_hf = Dataset.from_pandas(test_df_proteins_updated, preserve_index=False)

In [27]:
from datasets import DatasetDict

temp_hold_dataset = DatasetDict({
    "test": test_hf
})

In [28]:
temp_hold_dataset

DatasetDict({
    test: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'go_bp_created', 'go_mf_created', 'go_cc_created', 'date_added', 'last_modified', 'length', 'structure_path', 'interpro_ids', 'interpro_location', 'string_id', 'interaction_partners', 'ppi_formatted'],
        num_rows: 17815
    })
})

In [29]:
temp_hold_dataset.push_to_hub(
    "wanglab/cafa5",
    config_name="temporal_holdout_ext_2022_2025_reasoning",
    commit_message="Upload Temporal holdout data with new interpro annots"
)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/18 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        :  68%|######7   | 13.5MB / 19.9MB            

CommitInfo(commit_url='https://huggingface.co/datasets/wanglab/cafa5/commit/d352b9fd1f7cd36beff09dd770a88f1ec8829aba', commit_message='Upload Temporal holdout data with new interpro annots', commit_description='', oid='d352b9fd1f7cd36beff09dd770a88f1ec8829aba', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/wanglab/cafa5', endpoint='https://huggingface.co', repo_type='dataset', repo_id='wanglab/cafa5'), pr_revision=None, pr_num=None)

In [30]:
from datasets import Dataset

metadata_hf = Dataset.from_pandas(interpro_metadata_updated, preserve_index=False)

In [31]:
from datasets import DatasetDict

interpro_metadata_dataset = DatasetDict({
    "test": metadata_hf
})

In [32]:
interpro_metadata_dataset

DatasetDict({
    test: Dataset({
        features: ['interpro_id', 'entry_name', 'type'],
        num_rows: 14374
    })
})

In [33]:
interpro_metadata_dataset.push_to_hub(
    "wanglab/cafa5",
    config_name="interpro_temp_holdout_ext_22_25_metadata",
    commit_message="Upload Metadata with new interpro annots"
)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/15 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        : 100%|##########|  400kB /  400kB            

README.md:   0%|          | 0.00/42.3k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/wanglab/cafa5/commit/18c096aadb183ea7afb3a710ae466dfa02e0c932', commit_message='Upload Metadata with new interpro annots', commit_description='', oid='18c096aadb183ea7afb3a710ae466dfa02e0c932', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/wanglab/cafa5', endpoint='https://huggingface.co', repo_type='dataset', repo_id='wanglab/cafa5'), pr_revision=None, pr_num=None)